In [20]:
import jax
import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation

from uphate.uphate import pdist_squared

In [33]:
jnp.arange(1, 4)

Array([1, 2, 3], dtype=int32)

In [52]:
Y = jnp.array(
    [
        [1, 0],
        [0, 1],
        [1, 1],
        [0, 0],
    ],
    dtype=jnp.float32,
)
X = (
    jnp.concat([Y, jnp.ones((4, 1))], axis=1)
    @ Rotation.from_euler("xyz", jnp.array([90, 45, 20]), degrees=True).as_matrix()
)
X_alt = jnp.concat([Y, jnp.arange(1, 5, dtype=jnp.float32)[:, None]], axis=1)

(pdist_squared(X) - pdist_squared(Y)).round(4), (pdist_squared(X_alt) - pdist_squared(Y)).round(4), 

(Array([[ 0., -0., -0., -0.],
        [-0.,  0., -0., -0.],
        [-0., -0.,  0., -0.],
        [-0., -0., -0.,  0.]], dtype=float32),
 Array([[0., 1., 4., 9.],
        [1., 0., 1., 4.],
        [4., 1., 0., 1.],
        [9., 4., 1., 0.]], dtype=float32))

In [74]:
def mds_loss(X, Y):
    triu_indices = jnp.triu_indices_from(X, k=1)
    return ((pdist_squared(X)[triu_indices] ** 0.5 - pdist_squared(Y)[triu_indices] ** 0.5) ** 2).sum()

mds_loss_grad = jax.grad(mds_loss, argnums=(0, 1))

In [54]:
def pdist_squared(x):
    return jnp.sum((x[:, None] - x[None, :]) ** 2, axis=-1)

def f(x):
    return pdist_squared(x).sum()

In [77]:
from jaxopt import implicit_diff

implicit_diff.custom_root(jax.grad(mds_loss, argnums=(0, 1)))

<function jaxopt._src.implicit_diff.custom_root.<locals>.wrapper(solver_fun)>

In [72]:
jax.value_and_grad(mds_loss, argnums=(0, 1))(X_alt, Y)

(Array(1.8004574, dtype=float32),
 (Array([[ 0.36700684, -1.4725797 , -2.5781524 ],
         [-0.95279324,  0.36700684, -0.21877956],
         [ 0.5857864 ,  1.1055728 ,  2.796932  ],
         [ 0.        ,  0.        ,  0.        ]], dtype=float32),
  Array([[-0.4494897,  2.9216256],
         [ 1.2779168, -0.4494897],
         [-0.8284271, -2.472136 ],
         [ 0.       ,  0.       ]], dtype=float32)))

In [73]:
jax.value_and_grad(mds_loss, argnums=(0, 1))(X, Y)

(Array(3.1974423e-14, dtype=float32),
 (Array([[-4.2417970e-08, -4.2418005e-08, -3.2810073e-07],
         [ 2.2966843e-07,  2.2966849e-07,  2.9762455e-07],
         [-1.8725046e-07, -1.8725048e-07,  3.0476166e-08],
         [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00]], dtype=float32),
  Array([[ 1.6858739e-07, -2.8779669e-07],
         [-4.0700598e-07,  1.6858739e-07],
         [ 2.3841858e-07,  1.1920929e-07],
         [ 0.0000000e+00,  0.0000000e+00]], dtype=float32)))

In [56]:
jax.value_and_grad(f)(X_alt)

(Array(56., dtype=float32),
 Array([[  8.,  -8., -24.],
        [ -8.,   8.,  -8.],
        [  8.,   8.,   8.],
        [ -8.,  -8.,  24.]], dtype=float32))